# Python Bindings Development Roadmap

This notebook documents what the Rust Barter engine exposes vs. what the Python
bindings currently cover, and lays out a phased implementation plan to close the
gaps.

## Current State

| Feature Area | Python Status | Rust Status |
|---|---|---|
| Instruments & Indexing | **Complete** | Complete |
| Market Data Streaming (trades) | **Complete** (12 exchanges) | Complete (12+ exchanges) |
| Market Data (L1 books) | **Complete** (Binance, Kraken, Bybit) | Full `OrderBookMap` + manager |
| Market Data (L2 books) | Not exposed | Full `OrderBookMap` + manager |
| Trading Statistics | **Complete** | Complete |
| Engine / Backtesting | **Complete** (`run_backtest()` with custom strategy & risk) | Full lifecycle |
| Custom Strategies | **Complete** (Python callback via `strategy=`) | 4 traits, multi-strategy composition |
| Custom Risk Management | **Complete** (Python callback via `risk=`) | Full `RiskManager` trait |
| Order Management | **Complete** (`OrderRequestOpen`, `OrderRequestCancel`) | Full order types |
| Position Tracking | **Complete** (via `EngineState` snapshot) | `PositionManager`, `PositionExited` |
| Live Trading | **Not exposed** | Full `ExecutionClient` trait |
| Account Events | Partial (positions/balances visible in state) | `AccountEvent`, `AccountEventKind` |

---
## Phase 1: Strategy Callbacks — DONE

**Implemented in `barter-python` v0.2.0.**

### What was built

- `EngineState` snapshot type (`barter-python/src/state.rs`) exposing:
  - `state.price(instrument_index)` — current market price
  - `state.position(instrument_index)` — position with side, qty, entry price, PnL (realised + unrealised), trade count, timestamps
  - `state.instruments()` / `state.balances()` — full state dicts
  - `state.open_orders(instrument_index)` — active order count
- `OrderRequestOpen` and `OrderRequestCancel` types (`barter-python/src/order.rs`)
- `AlgoStrategy` trait delegated to Python callable via GIL acquisition per tick
- `run_backtest(config, data, strategy=my_fn)` parameter added

### Usage

```python
from barter import run_backtest, OrderRequestOpen

def my_strategy(state):
    price = state.price(0)
    if price and price < 100000:
        return [OrderRequestOpen(0, 0, "buy", price, 0.001)]
    return []

summary = await run_backtest(config, data, strategy=my_strategy)
```

---
## Phase 2: Risk Management Callbacks — DONE

**Implemented in `barter-python` v0.2.0.**

### What was built

- `RiskManager` trait delegated to Python callable (`barter-python/src/risk.rs`)
- Callback signature: `fn(state: EngineState, orders: List[OrderRequestOpen]) -> List[OrderRequestOpen]`
- Return only approved orders; omitted orders are rejected
- Default (no callback): approve all orders
- Python exceptions caught and logged, falls back to approve-all

### Usage

```python
def my_risk(state, orders):
    return [o for o in orders if o.price * o.quantity <= 100.0]

summary = await run_backtest(config, data, strategy=my_strategy, risk=my_risk)
```

---
## Phase 3: Position & Account Event Exposure — DONE

**Implemented in `barter-python` v0.2.0.**

### What was built

Position data enriched in `EngineState` snapshot:
- `pnl_realised`, `pnl_unrealised` — both realised and unrealised PnL
- `quantity_max` — peak position size
- `trade_count` — number of fills
- `time_enter`, `time_update` — position lifecycle timestamps
- `open_orders_count` — active orders per instrument

### Still TODO for this phase

- `on_fill` / `on_position_closed` event callbacks (requires custom `InstrumentData`)
- `AccountEvent` types exposed as Python objects
- Direct access to closed `PositionExited` list during backtest

---
## Phase 4: Extended Market Data — DONE

**Implemented in `barter-python` v0.2.0.**

### What was built

- L1 order book subscriptions: `Subscription("binance_spot", "btc", "usdt", data_kind="order_book_l1")`
- L1 supported on: Binance Spot, Binance Futures, Kraken, Bybit Spot, Bybit Perpetuals
- 8 new exchanges added (total 12): Kraken, Bitfinex, Bitmex, Bybit Spot,
  Bybit Perpetuals, Gateio Spot, Gateio Perpetuals, Gateio Futures
- `Streams::builder_multi()` used for unified `DataKind` output when mixing trades + L1

### Still TODO for this phase

- L2 order book depth via `OrderBookMap` with background update manager
- `PyOrderBook` with `bids` and `asks` as `[(price, qty), ...]`

---
## Phase 5: Live Trading (Priority: Lower)

**Goal**: Run the full engine with real exchange connections.

### Tasks

1. **Expose `SystemBuilder` with live clock and execution**
   - `run_live(config, strategy_fn, risk_fn)` — uses `LiveClock`, real WebSockets,
     and live execution clients
   - Config switches from `mocked_exchange` to real exchange credentials

2. **Expose engine control commands**
   - `system.enable_trading()` / `system.disable_trading()`
   - `system.cancel_orders(filter)` / `system.close_positions(filter)`
   - `system.shutdown()` → `TradingSummary`

3. **Expose `OnDisconnectStrategy` and `OnTradingDisabled`**
   - Optional callbacks: `on_disconnect(exchange)`, `on_trading_disabled()`
   - Defaults: cancel all orders on disconnect, no-op on disable

4. **Add audit stream exposure**
   - `system.audit_stream()` → async iterator of engine events
   - Useful for logging, monitoring, and debugging

### Files to modify
- `barter-python/src/engine.rs` — add `run_live()`, expose `System` wrapper
- `barter-python/src/strategy.rs` — add disconnect/disabled callbacks

### Definition of Done
- User can run a Python strategy against live exchange data
- Graceful shutdown returns a `TradingSummary`

---
## Implementation Status

```
                    DONE                         TODO
    ┌──────────────────────────────┬──────────────────────────┐
    │ Phase 1: Strategy Callbacks  │ Phase 5: Live Trading    │
    │ Phase 2: Risk Callbacks      │   - run_live()           │
    │ Phase 3: Position Tracking   │   - OnDisconnect         │
    │ Phase 4: Market Data (L1)    │   - OnTradingDisabled    │
    │   - 12 exchanges             │   - Audit stream         │
    │   - L1 order books           │                          │
    │   - Custom strategy/risk     │ Remaining Phase 3/4:     │
    │   - Enriched state snapshot  │   - on_fill callbacks    │
    │                              │   - L2 order books       │
    │                              │   - AccountEvent types   │
    └──────────────────────────────┴──────────────────────────┘
```

---
## Technical Notes

### GIL Considerations

Calling Python callbacks from the Rust engine requires acquiring the GIL on
each engine tick. For backtesting this is acceptable (the engine is
single-threaded and I/O-free). For live trading, consider:

- Releasing the GIL during Rust-only processing (`py.allow_threads()`)
- Batching multiple events before calling back into Python
- Using `tokio::spawn_blocking()` for the Python callback to avoid blocking
  the async runtime

### State Snapshot Pattern (Implemented)

The strategy and risk callbacks use the **snapshot** pattern: engine state is
copied into Python-owned objects before each callback, avoiding Rust borrow
lifetime issues across the FFI boundary.

```
Engine tick → copy state to PyEngineStateSnapshot → acquire GIL → call Python
           → collect returned orders → release GIL → feed orders to risk manager
```

### Error Handling

Python exceptions in callbacks are caught gracefully:
- **Strategy callback error**: logged to stderr, returns no orders (noop tick)
- **Risk callback error**: logged to stderr, approves all orders (fail-open)

This ensures the engine never panics due to Python errors.